# Day 22 — Time-series features: rolling stats, trends, missingness

Status: COMPLETE — `src/features.py` built 91 causal features for all 790,215
rows of set A in ~3.5 min; 5/5 hermetic tests pass; windows saved to
`data/windows_setA.parquet` (70 MB).

**Iron rule:** every feature at hour *t* uses only rows with `ICULOS <= t`
(trailing windows + `shift(1)` baselines). Three families per signal:
`mean/std` over 6h+24h (level + volatility), `delta_6h` vs trailing baseline
(deterioration is a *slope*), `miss` fractions (measurement frequency is signal).

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from features import add_window_features  # noqa: E402

df = pd.read_csv("../data/training_setA/p000001.psv", sep="|")
df["pid"] = "p000001"
out = add_window_features(df)
print(out[["ICULOS", "HR", "HR_mean_6h", "HR_std_6h", "HR_miss_6h",
           "HR_delta_6h"]].head(4).to_string())

    ICULOS    HR  HR_mean_6h  HR_std_6h  HR_miss_6h  HR_delta_6h
0       1   NaN         NaN   0.000000    1.000000          NaN
1       2  97.0        97.0   0.000000    0.500000          NaN
2       3  89.0        93.0   5.656854    0.333333         -8.0
3       4  90.0        92.0   4.358899    0.250000         -3.0


In [2]:
r0 = out.loc[0, ["Lactate", "Lactate_mean_6h", "Lactate_miss_6h"]]
print(f"Lactate row 0: value={r0['Lactate']} "
      f"mean={r0['Lactate_mean_6h']} miss={r0['Lactate_miss_6h']}")
print("NaN policy: raw gaps are NEVER filled — mean over all-NaN stays NaN,")
print("std is 0 (no variation observed), miss fraction records the gap.")
print("Trees (LightGBM, Day 23) split NaN natively, so absence stays informative")
print("instead of being smeared by forward-fill.")

Lactate row 0: value=nan mean=nan miss=1.0
NaN policy: raw gaps are NEVER filled — mean over all-NaN stays NaN,
std is 0 (no variation observed), miss fraction records the gap.
Trees (LightGBM, Day 23) split NaN natively, so absence stays informative
instead of being smeared by forward-fill.


In [3]:
import json
from pathlib import Path

s = json.loads(Path("../models/features_summary.json").read_text())
print(f"patients: {s['n_patients']} | rows: {s['n_rows']} | "
      f"features: {s['n_features']} ({s['n_columns_total']} cols total)")
print("raw NaN rates (worst labs): " + ", ".join(
    f"{k} {v * 100:.1f}" for k, v in s["raw_nan_rate_top"].items()) + " (%)")
print(f"build time: {s['elapsed_s']} s | output: data/windows_setA.parquet (70 MB)")

patients: 20336 | rows: 790215 | features: 91 (112 cols total)
raw NaN rates (worst labs): Bilirubin_total 98.8, Lactate 96.6,
  Platelets 93.5, Creatinine 93.4, WBC 92.5 (%)
build time: 207.5 s | output: data/windows_setA.parquet (70 MB)


## Design decisions (defend these in review)

1. **`min_periods=1` + miss flags instead of dropping short histories.** 21% of
   septic patients onset within 6h — dropping their early rows would delete the
   hardest, most valuable cases. Row 0 above shows the graceful degradation.
2. **`delta` excludes the current hour** (`shift(1)` baseline). Current-minus-mean
   including current would dilute the trend toward zero exactly when it matters.
3. **No forward-fill.** Filling would erase the missingness signal *and* smear
   stale values across deterioration boundaries. NaN + indicators instead.
4. **Whole-patient chunks** in the builder: rolling ops are grouped by pid, so a
   chunk split can never leak one patient's future into another's window
   (proven by `test_no_cross_patient_leak`).

## Handoff to Day 23 (baseline)

`windows_setA.parquet` is model-ready: each row = one patient-hour with 91
features + `SepsisLabel`. The baseline trains Logistic Regression on these rows
with a **patient-ID split** (never row shuffle) and excludes post-onset hours.